# vbOCP — Test 1: pipeline ROM (POD)

Notebook orchestratore per la parte ROM. Per ora: checkpoint di verifica della base POD (confronto tra prodotto scalare H1-seminorma e H1 completa) sugli snapshot gia' generati.

## 1. Setup

In [ ]:
import os

while not os.path.isdir('src') and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir('..')
assert os.path.isdir('src'), "src/ non trovata: verifica dove e' montata la repo vbOCP"

REPO_ROOT      = os.getcwd()
CONFIG_PATH    = os.path.join('configs', 'test1.yaml')
SNAPSHOTS_PATH = os.path.join('data', 'snapshots', 'test1_300.npz')
OUTPUT_DIR     = os.path.join('notebooks', 'output')

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('REPO_ROOT     :', REPO_ROOT)
print('SNAPSHOTS_PATH:', SNAPSHOTS_PATH)

## 2. Checkpoint: base POD con due prodotti scalari a confronto

Carica gli snapshot gia' generati (`data/snapshots/test1_300.npz`), assembla i due prodotti scalari (H1-seminorma = solo `A_diff`, H1 completa = `A_diff + M_full`), costruisce la base POD per stato e aggiunto con entrambi, e confronta visivamente il decadimento degli autovalori.

In [ ]:
import numpy as np
import yaml

from src.full_order.mesh import load_mesh
from src.full_order.assembly import assemble_operators
from src.rom.inner_product import assemble_full_mass_matrix
from src.rom.pod import build_pod_basis, plot_eigenvalue_decay

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

mesh_data = load_mesh(config['mesh']['path'], config['boundary_markers'])
operators = assemble_operators(mesh_data, config['problem']['omega_obs'])

A_diff = operators['A_diff']
M_full = assemble_full_mass_matrix(mesh_data)

X_seminorm = A_diff
X_full     = A_diff + M_full

print('Nh =', mesh_data['Nh'])

In [ ]:
data = np.load(SNAPSHOTS_PATH)
Y, P = data['Y'], data['P']
print('Y shape:', Y.shape, ' P shape:', P.shape)

N_MODES = 30

basis_y_semi, eig_y_semi = build_pod_basis(Y, X_seminorm, N_MODES)
basis_p_semi, eig_p_semi = build_pod_basis(P, X_seminorm, N_MODES)

basis_y_full, eig_y_full = build_pod_basis(Y, X_full, N_MODES)
basis_p_full, eig_p_full = build_pod_basis(P, X_full, N_MODES)

In [ ]:
plot_eigenvalue_decay(eig_y_semi, eig_p_semi,
                       output_path=os.path.join(OUTPUT_DIR, 'pod_decay_seminorm.png'))

In [ ]:
plot_eigenvalue_decay(eig_y_full, eig_p_full,
                       output_path=os.path.join(OUTPUT_DIR, 'pod_decay_full.png'))

In [ ]:
from IPython.display import Image, display

print('H1-seminorma (A_diff):')
display(Image(os.path.join(OUTPUT_DIR, 'pod_decay_seminorm.png')))

print('H1 completa (A_diff + M_full):')
display(Image(os.path.join(OUTPUT_DIR, 'pod_decay_full.png')))